# Data processing
## Step 1: Data Loading and basic check

import pandas as pd
import numpy as np
df = pd.read_csv("bank-additional-full.csv", sep=";")

In [27]:
print(df.head())
print(df.shape)

   age        job  marital    education  default housing loan    contact  \
0   56  housemaid  married     basic.4y       no      no   no  telephone   
1   57   services  married  high.school  unknown      no   no  telephone   
2   37   services  married  high.school       no     yes   no  telephone   
3   40     admin.  married     basic.6y       no      no   no  telephone   
4   56   services  married  high.school       no      no  yes  telephone   

  month day_of_week  ...  campaign  pdays  previous     poutcome emp.var.rate  \
0   may         mon  ...         1    999         0  nonexistent          1.1   
1   may         mon  ...         1    999         0  nonexistent          1.1   
2   may         mon  ...         1    999         0  nonexistent          1.1   
3   may         mon  ...         1    999         0  nonexistent          1.1   
4   may         mon  ...         1    999         0  nonexistent          1.1   

   cons.price.idx  cons.conf.idx  euribor3m  nr.employed

In [28]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

From this info, there are no null values. However, there exists "unknown" category.
Then check Categorical Values.

In [30]:
for col in df.select_dtypes(include="object").columns:
    print("\n", col)
    print(df[col].value_counts())


 job
job
admin.           10422
blue-collar       9254
technician        6743
services          3969
management        2924
retired           1720
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
unknown            330
Name: count, dtype: int64

 marital
marital
married     24928
single      11568
divorced     4612
unknown        80
Name: count, dtype: int64

 education
education
university.degree      12168
high.school             9515
basic.9y                6045
professional.course     5243
basic.4y                4176
basic.6y                2292
unknown                 1731
illiterate                18
Name: count, dtype: int64

 default
default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64

 housing
housing
yes        21576
no         18622
unknown      990
Name: count, dtype: int64

 loan
loan
no         33950
yes         6248
unknown      990
Name: count, dtype: int64

 contact
con

In [31]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 12


In [32]:
#Only 12 duplicate rows, delete them.
df = df.drop_duplicates().copy()
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (41176, 21)


In [33]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,41176.0,40.023800,10.420680,17.000,32.000,38.000,47.000,98.000
duration,41176.0,258.315815,259.305321,0.000,102.000,180.000,319.000,4918.000
campaign,41176.0,2.567879,2.770318,1.000,1.000,2.000,3.000,56.000
pdays,41176.0,962.464810,186.937102,0.000,999.000,999.000,999.000,999.000
previous,41176.0,0.173013,0.494964,0.000,0.000,0.000,0.000,7.000
emp.var.rate,41176.0,0.081922,1.570883,-3.400,-1.800,1.100,1.400,1.400
cons.price.idx,41176.0,93.575720,0.578839,92.201,93.075,93.749,93.994,94.767
cons.conf.idx,41176.0,-40.502863,4.627860,-50.800,-42.700,-41.800,-36.400,-26.900
euribor3m,41176.0,3.621293,1.734437,0.634,1.344,4.857,4.961,5.045
nr.employed,41176.0,5167.034870,72.251364,4963.600,5099.100,5191.000,5228.100,5228.100


## Step 2: Analysis based on checked info

In [37]:
print(df["y"].value_counts())
print(df["y"].value_counts(normalize=True))

y
no     36537
yes     4639
Name: count, dtype: int64
y
no     0.887337
yes    0.112663
Name: proportion, dtype: float64


In [38]:
target_distribution = (
    df["y"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(target_distribution)

y
no     88.73
yes    11.27
Name: proportion, dtype: float64


In [39]:
df["y"] = df["y"].map({
    "no": 0,
    "yes": 1
})

print(df["y"].value_counts())

y
0    36537
1     4639
Name: count, dtype: int64


0 = no subscription
1 = subscribed

In [40]:
'''
# 999 means never contacted before. So separate it into an individual column
# False = 0 True = 1
df["previously_contacted"] = (df["pdays"] != 999).astype(int)
df["pdays_clean"] = df["pdays"].replace(999, np.nan)
# pdays is separated into previously_contacted and pdays_clean
df = df.drop(columns=["pdays"])
print(df[["previously_contacted", "pdays_clean"]].head(20))
print(df["previously_contacted"].value_counts())
print(df["pdays_clean"].describe())
'''
# After checking the outcome, less than 10% of customers were previously contacted

'\n# 999 means never contacted before. So separate it into an individual column\n# False = 0 True = 1\ndf["previously_contacted"] = (df["pdays"] != 999).astype(int)\ndf["pdays_clean"] = df["pdays"].replace(999, np.nan)\n# pdays is separated into previously_contacted and pdays_clean\ndf = df.drop(columns=["pdays"])\nprint(df[["previously_contacted", "pdays_clean"]].head(20))\nprint(df["previously_contacted"].value_counts())\nprint(df["pdays_clean"].describe())\n'

In [41]:
df["previously_contacted"] = (df["pdays"] != 999).astype(int)

df["pdays_clean"] = np.where(
    df["pdays"] == 999,
    0,
    df["pdays"]
)

df = df.drop(columns=["pdays"])

In [24]:
#df_with_duration = df.copy()
#df_without_duration = df.drop(columns=["duration"]).copy()

Almost done, check again.

In [42]:
print("Final shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nTarget distribution:")
print(df["y"].value_counts())
print(df["y"].value_counts(normalize=True))

print("\nData types:")
print(df.dtypes)

Final shape: (41176, 22)

Missing values:
age                     0
job                     0
marital                 0
education               0
default                 0
housing                 0
loan                    0
contact                 0
month                   0
day_of_week             0
duration                0
campaign                0
previous                0
poutcome                0
emp.var.rate            0
cons.price.idx          0
cons.conf.idx           0
euribor3m               0
nr.employed             0
y                       0
previously_contacted    0
pdays_clean             0
dtype: int64

Duplicate rows:
0

Target distribution:
y
0    36537
1     4639
Name: count, dtype: int64
y
0    0.887337
1    0.112663
Name: proportion, dtype: float64

Data types:
age                       int64
job                      object
marital                  object
education                object
default                  object
housing                  object
loan          